# Lab 10: Test MongoDB, Ollama and the sandbox

Notebook for running inside the `comp40771` container.

It demonstrates how to:
- configure the MongoDB connection
- create collections and indexes
- save data
- load data
- update data
- delete data
- verify if ollama is running
- list the models
- runs an ollama query
- tests the sandbox container

The notebook assumes MongoDB is reachable through Docker Compose at `mongodb:27017`.

---

## 1. Install dependencies

Run the next cell only if `pymongo` is not already available in the container.

In [3]:
# Uncomment if needed
!pip install pymongo==4.10.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 977.0 kB/s  0:00:01ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pymongo]m1/2 [pymongo]


## 2. Configuration

In [4]:
import os
from pprint import pprint
from pymongo import MongoClient

MONGO_URI = os.getenv('MONGO_URI', 'mongodb://mongodb:27017')
MONGO_DB = os.getenv('MONGO_DB', 'lab10_audit')

print('Mongo URI:', MONGO_URI)
print('Database:', MONGO_DB)

Mongo URI: mongodb://mongodb:27017
Database: lab10_audit


## 3. Connect to MongoDB

In [5]:
client = MongoClient(MONGO_URI)
db = client[MONGO_DB]

print('Connected successfully')
print('Collections currently present:', db.list_collection_names())

Connected successfully
Collections currently present: ['sandbox_classifications', 'telemetry_logs', 'reports', 'sandbox_runs', 'agent_observations']


## 4. Create collections and indexes

In [6]:
required_collections = [
    'sandbox_runs',
    'telemetry_logs',
    'sandbox_classifications',
    'reports'
]

existing = set(db.list_collection_names())
for name in required_collections:
    if name not in existing:
        db.create_collection(name)

db['sandbox_runs'].create_index('run_id', unique=True)
db['telemetry_logs'].create_index('run_id')
db['sandbox_classifications'].create_index('run_id')
db['reports'].create_index('run_id')

print('Collections after setup:', db.list_collection_names())

Collections after setup: ['sandbox_classifications', 'telemetry_logs', 'reports', 'sandbox_runs', 'agent_observations']


## 5. Create / Save data

In [7]:
run_document = {
    'run_id': 'run_001',
    'api_id': 'api07',
    'started_at': '2026-03-15T10:00:00Z',
    'status': 'completed'
}

telemetry_document = {
    'run_id': 'run_001',
    'cpu_peak': 78.5,
    'network_attempts': 3,
    'file_changes': 2,
    'thread_count': 5
}

classification_document = {
    'run_id': 'run_001',
    'api_id': 'api07',
    'classification': 'suspicious',
    'confidence': 0.81,
    'explanation': 'Repeated outbound connections combined with file writes.'
}

report_document = {
    'run_id': 'run_001',
    'api_id': 'api07',
    'risk_score': 0.76,
    'classification': 'suspicious',
    'recommended_mitigation': 'Restrict network access and filesystem write scope.'
}

# Upsert avoids duplicate key errors if the notebook is run more than once
db['sandbox_runs'].update_one({'run_id': run_document['run_id']}, {'$set': run_document}, upsert=True)
db['telemetry_logs'].update_one({'run_id': telemetry_document['run_id']}, {'$set': telemetry_document}, upsert=True)
db['sandbox_classifications'].update_one({'run_id': classification_document['run_id']}, {'$set': classification_document}, upsert=True)
db['reports'].update_one({'run_id': report_document['run_id']}, {'$set': report_document}, upsert=True)

print('Documents saved successfully')

Documents saved successfully


## 6. Load / Read data

In [8]:
run = db['sandbox_runs'].find_one({'run_id': 'run_001'})
telemetry = db['telemetry_logs'].find_one({'run_id': 'run_001'})
classification = db['sandbox_classifications'].find_one({'run_id': 'run_001'})
report = db['reports'].find_one({'run_id': 'run_001'})

print('Run:')
pprint(run)
print('\nTelemetry:')
pprint(telemetry)
print('\nClassification:')
pprint(classification)
print('\nReport:')
pprint(report)

Run:
{'_id': ObjectId('69b6b77d10b27642e0122a3a'),
 'api_id': 'api07',
 'run_id': 'run_001',
 'started_at': '2026-03-15T10:00:00Z',
 'status': 'completed'}

Telemetry:
{'_id': ObjectId('69b6b77d10b27642e0122a3c'),
 'cpu_peak': 78.5,
 'file_changes': 2,
 'network_attempts': 3,
 'run_id': 'run_001',
 'thread_count': 5}

Classification:
{'_id': ObjectId('69b6b77d10b27642e0122a3e'),
 'api_id': 'api07',
 'classification': 'suspicious',
 'confidence': 0.81,
 'explanation': 'Repeated outbound connections combined with file writes.',
 'run_id': 'run_001'}

Report:
{'_id': ObjectId('69b6b77d10b27642e0122a40'),
 'api_id': 'api07',
 'classification': 'suspicious',
 'recommended_mitigation': 'Restrict network access and filesystem write '
                           'scope.',
 'risk_score': 0.76,
 'run_id': 'run_001'}


## 7. Query multiple documents

In [9]:
results = list(db['sandbox_classifications'].find({
    'classification': {'$in': ['suspicious', 'potentially_harmful']}
}))

print(f'Found {len(results)} suspicious or harmful classifications')
for item in results:
    pprint(item)

Found 1 suspicious or harmful classifications
{'_id': ObjectId('69b6b77d10b27642e0122a3e'),
 'api_id': 'api07',
 'classification': 'suspicious',
 'confidence': 0.81,
 'explanation': 'Repeated outbound connections combined with file writes.',
 'run_id': 'run_001'}


## 8. Update data

In [10]:
update_result = db['sandbox_classifications'].update_one(
    {'run_id': 'run_001'},
    {'$set': {
        'classification': 'potentially_harmful',
        'confidence': 0.93,
        'explanation': 'High-risk behaviour confirmed after additional telemetry review.'
    }}
)

print('Modified count:', update_result.modified_count)
pprint(db['sandbox_classifications'].find_one({'run_id': 'run_001'}))

Modified count: 1
{'_id': ObjectId('69b6b77d10b27642e0122a3e'),
 'api_id': 'api07',
 'classification': 'potentially_harmful',
 'confidence': 0.93,
 'explanation': 'High-risk behaviour confirmed after additional telemetry '
                'review.',
 'run_id': 'run_001'}


## 9. Update many documents

In [11]:
update_many_result = db['sandbox_runs'].update_many(
    {'status': 'completed'},
    {'$set': {'reviewed': True}}
)

print('Modified count:', update_many_result.modified_count)

Modified count: 1


## 10. Delete data

In [12]:
# Delete one example report
delete_one_result = db['reports'].delete_one({'run_id': 'run_001'})
print('Deleted from reports:', delete_one_result.deleted_count)

# Delete many benign classifications if any exist
delete_many_result = db['sandbox_classifications'].delete_many({'classification': 'benign'})
print('Deleted benign classifications:', delete_many_result.deleted_count)

Deleted from reports: 1
Deleted benign classifications: 0


## 11. Verify final state

In [13]:
for collection_name in required_collections:
    count = db[collection_name].count_documents({})
    print(f'{collection_name}: {count} document(s)')

sandbox_runs: 1 document(s)
telemetry_logs: 1 document(s)
sandbox_classifications: 1 document(s)
reports: 0 document(s)


## 12. Close the connection

In [14]:
client.close()
print('MongoDB connection closed')

MongoDB connection closed


## 13. Test ollama

In [15]:
import requests
import json

OLLAMA_BASE = "http://ollama:11434"

print("Ollama base URL:", OLLAMA_BASE)


Ollama base URL: http://ollama:11434


## 14. Check if Ollama server is reachable

In [16]:
try:
    r = requests.get(f"{OLLAMA_BASE}/")
    print("Status:", r.status_code)
    print("Response:", r.text[:200])
except Exception as e:
    print("Connection failed:", e)


Status: 200
Response: Ollama is running


## 15. List installed models

In [17]:
r = requests.get(f"{OLLAMA_BASE}/api/tags")
models = r.json()

print(json.dumps(models, indent=2))


{
  "models": [
    {
      "name": "phi3:mini",
      "model": "phi3:mini",
      "modified_at": "2026-03-15T13:43:37.086366037Z",
      "size": 2176178913,
      "digest": "4f222292793889a9a40a020799cfd28d53f3e01af25d48e06c5e708610fc47e9",
      "details": {
        "parent_model": "",
        "format": "gguf",
        "family": "phi3",
        "families": [
          "phi3"
        ],
        "parameter_size": "3.8B",
        "quantization_level": "Q4_0"
      }
    },
    {
      "name": "llama3.2:1b",
      "model": "llama3.2:1b",
      "modified_at": "2026-03-15T13:42:17.26709259Z",
      "size": 1321098329,
      "digest": "baf6a787fdffd633537aa2eb51cfd54cb93ff08e28040095462bb63daf552878",
      "details": {
        "parent_model": "",
        "format": "gguf",
        "family": "llama",
        "families": [
          "llama"
        ],
        "parameter_size": "1.2B",
        "quantization_level": "Q8_0"
      }
    }
  ]
}


## 16. Simple text generation test

In [18]:
payload = {
    "model": "llama3.2:1b",
    "prompt": "Write one sentence explaining MQTT.",
    "stream": False
}

response = requests.post(
    f"{OLLAMA_BASE}/api/generate",
    json=payload,
    timeout=120
)

data = response.json()
print(data["response"])


MQTT (Message Queuing Telemetry Transport) is a lightweight, publish-subscribe messaging protocol that allows devices to send and receive messages over the internet with high reliability and low latency.


## 17. Test both models used in the lab

In [19]:
models = ["llama3.2:1b", "phi3:mini"]

for model in models:
    print("\nTesting model:", model)
    
    payload = {
        "model": model,
        "prompt": "Explain CI/CD in one sentence.",
        "stream": False
    }
    
    r = requests.post(f"{OLLAMA_BASE}/api/generate", json=payload)
    print(r.json()["response"])



Testing model: llama3.2:1b
CI/CD stands for Continuous Integration and Continuous Deployment, a software development process that automates the build, test, and deployment of software changes to various environments, such as production and staging, in a rapid and repeatable manner.

Testing model: phi3:mini
CI/CD stands for Continuous Integration and Continuous Deployment, a software development methodology that facilitates automated testing and deployment of code changes frequently to improve efficiency and reduce the risk of project failure by detecting errors as soon as they occur.


## 18. Measure response latency

In [20]:
import time

model = "llama3.2:1b"

payload = {
    "model": model,
    "prompt": "List three advantages of containerisation.",
    "stream": False
}

start = time.time()

r = requests.post(f"{OLLAMA_BASE}/api/generate", json=payload)

end = time.time()

print("Latency:", round(end - start, 2), "seconds")
print(r.json()["response"])


Latency: 14.95 seconds
Here are three advantages of containerization:

1. **Portability and Reusability**: Containers are lightweight, easy to move, and can be easily reused on different machines or environments. This makes them ideal for deploying applications in cloud environments, allowing for greater flexibility and scalability.

2. **Isolation and Security**: Containers provide a high level of isolation between applications, which helps prevent resource leaks, data corruption, and security breaches. This is achieved through techniques like networking, process control, and access control.

3. **Simplified Deployment and Management**: Containers make it easier to deploy and manage applications across different environments, such as development, testing, staging, and production. The lightweight nature of containers reduces the complexity and overhead associated with virtualization, making it simpler for teams to deploy and maintain applications on various platforms.


## Interact with the sandbox api
Starter code below provides a minimal Python client for interacting with the sandbox REST API. It assumes the sandbox container is reachable at http://sandbox:8000, which matches the environment variable configuration in your compose file. Example structure suitable for a notebook or a small Python module.

In [23]:
import requests
import json
import time

SANDBOX_BASE = "http://sandbox:8000"

print("Sandbox base:", SANDBOX_BASE)
print("-" * 60)

# ---------------------------------------------------
# Connectivity check
# ---------------------------------------------------

urls = [
    f"{SANDBOX_BASE}/",
    f"{SANDBOX_BASE}/docs",
    f"{SANDBOX_BASE}/openapi.json"
]

for url in urls:
    try:
        r = requests.get(url, timeout=3)
        print(url, "->", r.status_code)
    except Exception as e:
        print(url, "-> failed:", e)

print("-" * 60)


# ---------------------------------------------------
# Helper functions
# ---------------------------------------------------

def api_get(path):
    url = f"{SANDBOX_BASE}{path}"
    r = requests.get(url, timeout=10)

    if r.status_code >= 400:
        print("GET error:", r.status_code, r.text)

    try:
        return r.json()
    except Exception:
        return r.text


def api_post(path, payload=None):
    url = f"{SANDBOX_BASE}{path}"

    r = requests.post(url, json=payload or {}, timeout=10)

    if r.status_code == 409:
        print(f"{path} -> already active (409 conflict)")
        return {"status": "conflict"}

    if r.status_code >= 400:
        print("POST error:", r.status_code, r.text)

    try:
        return r.json()
    except Exception:
        return r.text


# ---------------------------------------------------
# Start telemetry
# ---------------------------------------------------

print("Starting telemetry...")
resp = api_post("/telemetry/start", {})
print(resp)

print("-" * 60)


# ---------------------------------------------------
# Run execution test
# ---------------------------------------------------

payload = {
    "api_id": "api01",
    "duration": 5
}

print("Running sandbox execution...")
resp = api_post("/execute", payload)
print(json.dumps(resp, indent=2))

print("-" * 60)


# ---------------------------------------------------
# Poll telemetry results
# ---------------------------------------------------

print("Polling telemetry status...")

end_time = time.time() + 15

while time.time() < end_time:

    try:
        data = api_get("/telemetry/status")
        print(json.dumps(data, indent=2))
    except Exception as e:
        print("Polling failed:", e)

    time.sleep(3)

print("Polling finished.")


Sandbox base: http://sandbox:8000
------------------------------------------------------------
http://sandbox:8000/ -> 200
http://sandbox:8000/docs -> 200
http://sandbox:8000/openapi.json -> 200
------------------------------------------------------------
Starting telemetry...
/telemetry/start -> already active (409 conflict)
{'status': 'conflict'}
------------------------------------------------------------
Running sandbox execution...
POST error: 404 {"detail":"Not Found"}
{
  "detail": "Not Found"
}
------------------------------------------------------------
Polling telemetry status...
{
  "running": true,
  "config": {
    "monitor_threads": 4,
    "monitored_ports": [
      8501,
      8502,
      8503
    ],
    "packet_compare_pairs": [
      [
        8501,
        8502
      ],
      [
        8501,
        8503
      ]
    ],
    "network_sample_interval_s": 0.5,
    "file_sample_interval_s": 1.0,
    "cpu_sample_interval_s": 1.0,
    "thread_sample_interval_s": 1.0,
    "st

# Sandbox API Interaction Script -- Explanation

Code in the notebook cell demonstrates a minimal Python client for
interacting with the Sandbox REST API used in the lab environment. The
script performs four main tasks: verifying connectivity, defining helper
functions for API communication, starting telemetry monitoring, and
executing a test workload while collecting telemetry data.

## 1. Configuration and Imports

The script begins by importing required Python libraries:

-   **requests** -- used to communicate with the REST API.
-   **json** -- formats responses into readable JSON output.
-   **time** -- controls polling intervals.

The variable `SANDBOX_BASE` defines the base URL of the sandbox API:

    SANDBOX_BASE = "http://sandbox:8000"

Inside the Docker network, the hostname `sandbox` resolves to the
sandbox container.

## 2. Connectivity Check

The first block verifies that the API is reachable. Three endpoints are
tested:

-   `/` -- basic root endpoint\
-   `/docs` -- Swagger documentation generated by FastAPI\
-   `/openapi.json` -- OpenAPI specification describing the API

Each endpoint is queried with a short timeout to confirm that the
service is running.

Purpose of this step is to quickly diagnose networking issues before
attempting more complex API calls.

## 3. REST Helper Functions

Two helper functions simplify interaction with the API.

### `api_get(path)`

Function sends HTTP GET requests to retrieve information from the
sandbox.

Steps performed:

1.  Construct full URL from `SANDBOX_BASE` and the endpoint path.
2.  Send request using `requests.get`.
3.  Print errors if HTTP status ≥ 400.
4.  Attempt to parse the response as JSON.
5.  If JSON parsing fails, return raw text.

This function is used for reading sandbox state such as telemetry
status.

### `api_post(path, payload=None)`

Function sends POST requests to trigger sandbox operations.

Steps performed:

1.  Construct the endpoint URL.
2.  Send POST request with optional JSON payload.
3.  Detect **409 Conflict** responses.

A 409 error occurs when the requested operation is already active.\
For example, starting telemetry when telemetry is already running.

Instead of stopping execution, the function prints a message and
continues.


## 4. Starting Telemetry Monitoring

Telemetry is activated with the endpoint:

    /telemetry/start

Telemetry monitoring records behavioural signals such as:

-   filesystem activity
-   process creation
-   network access
-   resource usage

Information collected here can later be used for behavioural analysis
and adversarial robustness testing.


## 5. Executing a Sandbox Task

Example workload is submitted using:

    /execute

Example payload used in the script:

    {
        "api_id": "api01",
        "duration": 5
    }

Typical meanings:

-   **api_id** -- identifier of the monitored API or test scenario
-   **duration** -- execution time of the monitored task

Sandbox executes the workload while telemetry continues collecting
behavioural signals.


## 6. Polling Telemetry Status

The final section repeatedly queries:

    /telemetry/status

Polling loop runs for a fixed duration and retrieves telemetry snapshots
periodically.

Polling is a common monitoring strategy where the client periodically
requests system state instead of maintaining a persistent connection.


## 7. Overall Workflow

The script implements the following pipeline:

1.  Verify sandbox API availability.
2.  Enable telemetry monitoring.
3.  Execute a sandbox workload.
4.  Poll telemetry data generated during execution.

This simple workflow forms the basis for more advanced experiments such
as behavioural monitoring, anomaly detection, and adversarial robustness
testing in later stages of the lab.
